# 3.26 — Gradient Boosting Machines

Gradient Boosting Machines (GBMs) build a prediction rule one small correction at a time: start with a simple baseline, find the error signal left behind by the current ensemble, fit a weak learner to that signal, and add it with shrinkage. In this lesson, every step is written from scratch with NumPy so the additive formula, empirical risk, cost penalty, validation gap, and stability tradeoff stay visible.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build gradient boosting one idea at a time. Run each cell in order and read the printed intermediate values — every residual, score, and model-selection number is exposed so the ensemble never feels like a black box. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, masks, and small numerical checks for boosting from scratch.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for the tiny synthetic data.

### 1. The additive model starts from a constant

Gradient boosting does not jump straight to a complicated model. It begins with a constant prediction $F_0(x)$, usually the value that minimizes the chosen loss before any features are used. For squared error, that constant is the mean of the training targets, because the derivative of $\sum_i(y_i-c)^2$ with respect to $c$ is $-2\sum_i(y_i-c)$, which is zero exactly when $c=\bar y$.

In [ ]:
x_w = np.array([0., 1., 2., 3., 4., 5.])  # one feature, sorted so tiny tree splits are easy to inspect.
y_w = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2])  # a small regression target with two visible regions.
F0_w = float(np.mean(y_w))  # squared-error best constant prediction.
print("targets:", y_w)  # inspect the observed values.
print("F0 = mean(y):", round(F0_w, 3))  # the initial ensemble predicts this everywhere.
assert round(F0_w, 3) == 2.55  # (1.0+1.4+1.7+3.2+3.8+4.2)/6.

▶ What you'll see: the starting prediction is 2.55 for every row, even though the left side targets are lower and the right side targets are higher.

In [ ]:
F_w = np.full_like(y_w, F0_w, dtype=float)  # current ensemble predictions after zero trees.
res0_w = y_w - F_w  # squared-error residuals, also the negative gradients.
print("initial predictions:", np.round(F_w, 3))  # inspect the flat model.
print("initial residuals:", np.round(res0_w, 3))  # errors the next learner must explain.

▶ What you'll see: negative residuals on the left and positive residuals on the right, so the first correction should lower left predictions and raise right predictions.

In [ ]:
plt.figure(figsize=(4.6, 3.2))
plt.scatter(x_w, y_w, color="black", label="target y")
plt.plot(x_w, F_w, color="crimson", label="constant F0")
plt.vlines(x_w, F_w, y_w, color="gray", alpha=0.7, label="residual")
plt.title("1: constant start leaves structured residuals")
plt.xlabel("x"); plt.ylabel("prediction / target"); plt.legend(); plt.show()

▶ What you'll see: the red horizontal line misses low points upward and high points downward; the gray vertical gaps are exactly what boosting will chase.

*Why it's done this way:* the constant is the safest first model because it solves the empirical-risk problem before features enter. For squared loss, using the mean makes positive and negative residuals sum to zero, so any later learner must explain structure in $x$, not a leftover global offset.

### 2. The next learner fits the negative gradient

For squared error $L_i=(y_i-F(x_i))^2$, the derivative with respect to the current prediction is $\partial L_i/\partial F=-2(y_i-F(x_i))$. The direction that lowers loss is the negative gradient, proportional to $y_i-F(x_i)$. That means a GBM can fit the residuals for squared error; for other losses, it fits the appropriate pseudo-residual instead.

In [ ]:
grad_w = -2 * (y_w - F_w)  # derivative of squared error with respect to prediction F.
neg_grad_w = -grad_w / 2  # scaled negative gradient; the factor 2 can be absorbed into the step size.
print("gradient dL/dF:", np.round(grad_w, 3))  # direction of steepest increase in loss.
print("negative-gradient target:", np.round(neg_grad_w, 3))  # direction a learner should fit.
assert np.allclose(neg_grad_w, res0_w)  # for squared loss, pseudo-residuals equal residuals.

▶ What you'll see: the negative-gradient targets equal the residuals, so fitting residuals is gradient descent in function space.

In [ ]:
plt.figure(figsize=(4.6, 3.0))
plt.axhline(0, color="black", linewidth=0.8)
plt.bar(x_w, neg_grad_w, width=0.55, color=["steelblue" if r > 0 else "orange" for r in neg_grad_w])
plt.title("2: pseudo-residuals are the descent target")
plt.xlabel("x"); plt.ylabel("y - F(x)"); plt.show()

▶ What you'll see: bars below zero ask the next learner to subtract prediction on the left; bars above zero ask it to add prediction on the right.

*Why it's done this way:* boosting is gradient descent where the parameter is the whole prediction function $F$, not a finite vector of coefficients. A weak learner $h_t(x)$ is chosen because it approximates the negative-gradient direction using the allowed model class.

### 3. A one-split tree is a weak learner

A regression stump chooses one threshold and predicts a constant value on each side. When fitting residuals, the best leaf value under squared error is the mean residual inside that leaf, again because a mean minimizes squared deviations within a group. We can search all midpoint thresholds by hand.

In [ ]:
thresholds_w = (x_w[:-1] + x_w[1:]) / 2  # candidate split points between sorted feature values.
print("candidate thresholds:", thresholds_w)  # the stump will test each possible midpoint.

▶ What you'll see: five possible one-split trees for six sorted points.

In [ ]:
sse_by_split_w = []  # store residual-fitting SSE for each candidate stump.
leaf_values_w = []  # store left/right residual means for each split.
for thr_w in thresholds_w:
    left_w = x_w <= thr_w  # rows routed to the left leaf.
    right_w = ~left_w  # rows routed to the right leaf.
    lv_w = float(np.mean(res0_w[left_w]))  # squared-error best left correction.
    rv_w = float(np.mean(res0_w[right_w]))  # squared-error best right correction.
    pred_res_w = np.where(left_w, lv_w, rv_w)  # stump prediction for residuals.
    sse_w = float(np.sum((res0_w - pred_res_w) ** 2))  # residual-fitting error.
    leaf_values_w.append((lv_w, rv_w)); sse_by_split_w.append(sse_w)
print("split SSE:", np.round(sse_by_split_w, 3))  # lower means a better residual learner.

▶ What you'll see: the split between x=2 and x=3 gives the lowest residual SSE.

In [ ]:
best_idx_w = int(np.argmin(sse_by_split_w))  # choose the stump with lowest residual SSE.
best_thr_w = float(thresholds_w[best_idx_w])  # threshold selected by empirical residual fit.
left_val_w, right_val_w = leaf_values_w[best_idx_w]  # correction constants for the two leaves.
h1_w = np.where(x_w <= best_thr_w, left_val_w, right_val_w)  # first weak learner outputs.
print("best threshold:", best_thr_w)  # expected 2.5.
print("leaf corrections:", round(left_val_w, 3), round(right_val_w, 3))  # left negative, right positive.
assert best_thr_w == 2.5 and round(left_val_w, 3) == -1.183 and round(right_val_w, 3) == 1.183

▶ What you'll see: the stump learns to subtract about 1.183 on the left and add about 1.183 on the right.

In [ ]:
plt.figure(figsize=(4.8, 3.1))
plt.scatter(x_w, res0_w, color="black", label="residual target")
plt.step(x_w, h1_w, where="mid", color="seagreen", label="stump h1")
plt.axvline(best_thr_w, color="gray", linestyle="--", label="split")
plt.title("3: stump fit to residuals")
plt.xlabel("x"); plt.ylabel("correction"); plt.legend(); plt.show()

▶ What you'll see: the green two-level function approximates the residual pattern instead of the original target directly.

*Why it's done this way:* each leaf mean is the local best additive correction under squared loss. The stump is intentionally weak: it captures one broad pattern, leaving later trees to handle whatever residual structure remains.

### 4. Shrinkage adds the learner cautiously

The core GBM update is $F_t(x)=F_{t-1}(x)+\eta h_t(x)$. The learning rate $\eta$ (shrinkage) scales the new tree before adding it. A full step may reduce training loss faster, but a smaller step often generalizes better because each learner only makes a cautious correction.

In [ ]:
eta_w = 0.4  # shrinkage: add only 40% of the stump correction.
F1_w = F_w + eta_w * h1_w  # additive model after one weak learner.
loss0_w = float(np.mean((y_w - F_w) ** 2))  # empirical squared loss before update.
loss1_w = float(np.mean((y_w - F1_w) ** 2))  # empirical squared loss after shrinkage.
print("loss before:", round(loss0_w, 3), "loss after:", round(loss1_w, 3))
assert loss1_w < loss0_w  # the correction moves down the training loss.

▶ What you'll see: the empirical loss drops after adding only a fraction of the stump.

In [ ]:
full_step_w = F_w + h1_w  # eta=1 comparison.
loss_full_w = float(np.mean((y_w - full_step_w) ** 2))  # training loss for the aggressive step.
print("eta=0.4 loss:", round(loss1_w, 3), "eta=1.0 loss:", round(loss_full_w, 3))  # full step fits this toy faster.

▶ What you'll see: the full step has lower immediate training loss, but the lesson's stabilizing idea is not to chase this number alone.

In [ ]:
plt.figure(figsize=(4.8, 3.2))
plt.scatter(x_w, y_w, color="black", label="target")
plt.plot(x_w, F_w, color="gray", label="F0")
plt.step(x_w, F1_w, where="mid", color="seagreen", label="F1 = F0 + ηh1")
plt.step(x_w, full_step_w, where="mid", color="crimson", linestyle="--", label="η=1")
plt.title("4: shrinkage makes a cautious additive step")
plt.xlabel("x"); plt.ylabel("prediction"); plt.legend(); plt.show()

▶ What you'll see: the shrinkage step moves in the right direction without jumping all the way to the stump's piecewise averages.

*Why it's done this way:* $\eta$ is a regularization knob in function space. Many small steps let the ensemble refine the descent path and reduce the chance that one noisy tree permanently dominates the model.

### 5. The selection score includes cost, not just raw fit

The lesson's verified arithmetic uses three per-example losses: 0.268, 0.083, and 0.539. Their average is the empirical risk $R_S=0.297$. But model selection should use the full decision score, so a complexity, regularization, or operational cost of 0.070 is added.

In [ ]:
losses_toy_w = np.array([0.268, 0.083, 0.539])  # verified per-example losses from the lesson block.
R_S_w = float(np.mean(losses_toy_w))  # empirical risk = average training loss.
cost_w = 0.070  # method cost / regularization / operational penalty.
score_w = R_S_w + cost_w  # decision score used for selection.
print("empirical risk:", round(R_S_w, 3))  # 0.297.
print("score with cost:", round(score_w, 3))  # 0.367.
assert round(R_S_w, 3) == 0.297 and round(score_w, 3) == 0.367

▶ What you'll see: the raw training average is smaller than the score that should drive selection.

In [ ]:
plt.figure(figsize=(4.6, 3.0))
plt.bar(["raw risk", "+ cost", "final score"], [R_S_w, cost_w, score_w], color=["steelblue", "orange", "seagreen"])
plt.title("5: raw fit plus method cost")
plt.ylabel("score component"); plt.show()

▶ What you'll see: the final score is not just the empirical risk; the orange cost is part of the decision.

*Why it's done this way:* a flexible ensemble can keep lowering training loss by adding more trees, deeper trees, or larger steps. The cost term is the guardrail that makes the comparison about durable performance rather than memorized convenience.

### 6. Compare alternatives by gap and stability

A tempting alternative has decision score 0.407. The baseline score 0.367 is lower by 0.040, a relative gap of about 9.8%. If a stabilizing knob reduces the baseline score by 20%, the new stabilized score is 0.294, which wins the final three-way comparison.

In [ ]:
alt_score_w = 0.407  # a more flexible alternative from the lesson block.
score_for_decision_w = round(score_w, 3)  # use the displayed lesson score for the checked comparison.
gap_w = alt_score_w - score_for_decision_w  # absolute evidence for preferring the baseline over the alternative.
rel_gap_w = gap_w / alt_score_w  # gap as a fraction of the alternative score.
stable_w = 0.80 * score_for_decision_w  # stabilizing knob reduces the decision score by 20%.
print("gap:", round(gap_w, 3), "relative gap:", round(rel_gap_w, 3))
print("stabilized score:", round(stable_w, 3))
assert round(gap_w, 3) == 0.040 and round(rel_gap_w, 3) == 0.098 and round(stable_w, 3) == 0.294

▶ What you'll see: the absolute gap is small, so the relative gap gives a more honest sense of how strong the preference is.

In [ ]:
scores_w = np.array([score_for_decision_w, alt_score_w, stable_w])  # baseline, flexible alternative, stabilized model.
labels_w = ["baseline", "flexible", "stabilized"]  # names for the final decision.
winner_w = labels_w[int(np.argmin(scores_w))]  # lower score wins.
print("scores:", dict(zip(labels_w, np.round(scores_w, 3))))
print("carry forward:", winner_w)
assert winner_w == "stabilized"  # min(0.367, 0.407, 0.294) = 0.294.

▶ What you'll see: the stabilized model has the lowest full decision score.

In [ ]:
plt.figure(figsize=(4.8, 3.0))
colors_w = ["gray", "crimson", "seagreen"]
plt.bar(labels_w, scores_w, color=colors_w)
plt.ylabel("decision score (lower is better)")
plt.title("6: final GBM model-selection score")
plt.show()

▶ What you'll see: the green stabilized bar is lowest; the more flexible red bar is not automatically preferred.

*Why it's done this way:* boosting is powerful enough that raw training gains can be misleading. The final choice should compare scores on the same scale, include costs, and treat small gaps cautiously because they may vanish under resampling noise.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, masks, losses, and tiny from-scratch tree searches.
import matplotlib.pyplot as plt # load Matplotlib for inspecting residuals, step functions, and validation curves.
np.random.seed(0) # make all stochastic examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Start with the squared-error constant

**Goal.** Compute the initial constant model, because squared-error boosting starts from the mean target before adding trees. We build it in 2 steps.

In [ ]:
x_b1 = np.array([0., 1., 2., 3., 4., 5.]) # define one sorted feature so later stump splits are easy to inspect.
y_b1 = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2]) # define the regression target values.
print("x:", x_b1) # inspect the feature positions.
print("y:", y_b1) # inspect the targets before modeling.

▶ What you'll see: a small dataset whose target values are lower on the left and higher on the right.

In [ ]:
F0_b1 = float(np.mean(y_b1)) # compute the best constant under squared error.
F_b1 = np.full_like(y_b1, F0_b1, dtype=float) # predict the same mean for every row.
print("F0:", round(F0_b1, 3)) # inspect the initial prediction.
assert round(F0_b1, 3) == 2.55 # verify the hand-checkable mean.
plt.figure(figsize=(4, 3)) # create a compact first-model plot.
plt.scatter(x_b1, y_b1, color="black", label="target") # plot observed targets.
plt.plot(x_b1, F_b1, color="crimson", label="constant F0") # plot the constant model.
plt.title("Basic 1: constant initialization") # title the plot.
plt.xlabel("x"); plt.ylabel("y / prediction"); plt.legend(); plt.show() # label and display.

▶ What you'll see: one flat red line through the average of the target values.

👀 Takeaway: the initial GBM prediction is already the empirical-risk minimizer among constant functions.

### Basic 2 — Compute residuals as the next target

**Goal.** Find what the current model still misses, because squared-error boosting fits each new learner to residuals. We build it in 2 steps.

In [ ]:
x_b2 = np.array([0., 1., 2., 3., 4., 5.]) # recreate the toy feature values locally.
y_b2 = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2]) # recreate the target values locally.
F_b2 = np.full_like(y_b2, np.mean(y_b2), dtype=float) # build the constant prediction vector.
print("current predictions:", np.round(F_b2, 3)) # inspect the model before residual fitting.

▶ What you'll see: every row receives the same prediction, 2.55.

In [ ]:
res_b2 = y_b2 - F_b2 # compute residuals y - F(x), the squared-error negative gradients.
print("residuals:", np.round(res_b2, 3)) # inspect the signal for the next learner.
assert round(float(np.sum(res_b2)), 10) == 0.0 # residuals around the mean sum to zero.
plt.figure(figsize=(4, 3)) # create a residual bar chart.
plt.axhline(0, color="black", linewidth=0.8) # show the no-correction baseline.
plt.bar(x_b2, res_b2, color=["orange" if r < 0 else "steelblue" for r in res_b2]) # plot residual direction and size.
plt.title("Basic 2: residual targets") # title the plot.
plt.xlabel("x"); plt.ylabel("y - F0"); plt.show() # label and display.

▶ What you'll see: left residuals are negative and right residuals are positive.

👀 Takeaway: residuals tell the next tree where to subtract and where to add prediction.

### Basic 3 — Connect residuals to gradients

**Goal.** Verify that residual fitting is gradient descent for squared error, because GBMs generalize this idea to many losses. We build it in 2 steps.

In [ ]:
y_b3 = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2]) # define targets.
F_b3 = np.full_like(y_b3, np.mean(y_b3), dtype=float) # define current predictions.
grad_b3 = -2 * (y_b3 - F_b3) # compute d/dF of (y-F)^2 for each example.
print("gradient:", np.round(grad_b3, 3)) # inspect the steepest-increase direction.

▶ What you'll see: signs are opposite the residual signs because gradients point uphill.

In [ ]:
neg_grad_b3 = -grad_b3 / 2 # remove the constant factor because learning rate can absorb it.
res_b3 = y_b3 - F_b3 # compute ordinary residuals.
print("scaled negative gradient:", np.round(neg_grad_b3, 3)) # inspect descent targets.
assert np.allclose(neg_grad_b3, res_b3) # for squared loss, they match exactly.
plt.figure(figsize=(4, 3)) # create a comparison plot.
plt.plot(res_b3, marker="o", label="residual") # plot residuals.
plt.plot(neg_grad_b3, marker="x", linestyle="--", label="negative gradient") # plot pseudo-residuals.
plt.title("Basic 3: residual = negative gradient") # title the plot.
plt.legend(); plt.show() # display with labels.

▶ What you'll see: the two curves overlap perfectly.

👀 Takeaway: for squared error, a GBM fitting residuals is doing gradient descent in prediction space.

### Basic 4 — Search stump split candidates

**Goal.** Enumerate possible one-split trees, because a regression stump is the simplest weak learner. We build it in 2 steps.

In [ ]:
x_b4 = np.array([0., 1., 2., 3., 4., 5.]) # sorted feature values.
thresholds_b4 = (x_b4[:-1] + x_b4[1:]) / 2 # candidate splits between adjacent values.
print("thresholds:", thresholds_b4) # inspect split candidates.

▶ What you'll see: five midpoint thresholds are available.

In [ ]:
plt.figure(figsize=(4, 3)) # create a split-location plot.
plt.scatter(x_b4, np.zeros_like(x_b4), color="black", label="rows") # show data row positions.
for thr_b4 in thresholds_b4: # draw every possible threshold.
    plt.axvline(thr_b4, color="gray", alpha=0.5) # mark a candidate split.
plt.yticks([]) # remove irrelevant y ticks.
plt.title("Basic 4: possible stump thresholds") # title the plot.
plt.xlabel("x"); plt.show() # label and display.

▶ What you'll see: a threshold can only separate the sorted rows at midpoint gaps.

👀 Takeaway: a stump is weak because it can make only one binary partition of the feature axis.

### Basic 5 — Fit leaf values by averaging residuals

**Goal.** Compute the best constant correction in each leaf, because squared-error leaf predictions are means. We build it in 3 steps.

In [ ]:
x_b5 = np.array([0., 1., 2., 3., 4., 5.]) # define feature positions.
y_b5 = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2]) # define targets.
res_b5 = y_b5 - np.mean(y_b5) # compute residuals from the constant model.
thr_b5 = 2.5 # choose the natural split between the low and high regions.
left_b5 = x_b5 <= thr_b5 # mark rows routed left.
print("left rows:", left_b5.astype(int)) # inspect the partition.

▶ What you'll see: the first three rows go left and the last three rows go right.

In [ ]:
left_value_b5 = float(np.mean(res_b5[left_b5])) # best correction for the left leaf.
right_value_b5 = float(np.mean(res_b5[~left_b5])) # best correction for the right leaf.
print("leaf values:", round(left_value_b5, 3), round(right_value_b5, 3)) # inspect residual means.
assert round(left_value_b5, 3) == -1.183 and round(right_value_b5, 3) == 1.183 # verify leaf means.

In [ ]:
h_b5 = np.where(left_b5, left_value_b5, right_value_b5) # create the stump correction vector.
plt.figure(figsize=(4, 3)) # create a correction plot.
plt.scatter(x_b5, res_b5, color="black", label="residual") # show residual targets.
plt.step(x_b5, h_b5, where="mid", color="seagreen", label="leaf means") # show fitted leaf values.
plt.title("Basic 5: leaf means fit residuals") # title the plot.
plt.legend(); plt.show() # display.

▶ What you'll see: each leaf predicts the average residual of the points it owns.

👀 Takeaway: a regression-tree leaf is a local average correction under squared error.

### Basic 6 — Choose the best stump by residual SSE

**Goal.** Pick the split with the lowest residual-fitting error, because the weak learner should approximate the negative gradient as well as it can. We build it in 3 steps.

In [ ]:
x_b6 = np.array([0., 1., 2., 3., 4., 5.]) # define features.
y_b6 = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2]) # define targets.
res_b6 = y_b6 - np.mean(y_b6) # compute residual targets.
thresholds_b6 = (x_b6[:-1] + x_b6[1:]) / 2 # generate all stump thresholds.
print("candidate count:", len(thresholds_b6)) # inspect search size.

▶ What you'll see: there are five candidate stumps.

In [ ]:
sse_b6 = [] # store one residual SSE per split.
for thr_b6 in thresholds_b6: # evaluate every candidate threshold.
    left_b6 = x_b6 <= thr_b6 # route rows left or right.
    pred_b6 = np.where(left_b6, np.mean(res_b6[left_b6]), np.mean(res_b6[~left_b6])) # use leaf means.
    sse_b6.append(float(np.sum((res_b6 - pred_b6) ** 2))) # score residual fit.
print("SSE by threshold:", np.round(sse_b6, 3)) # inspect split quality.

In [ ]:
best_thr_b6 = float(thresholds_b6[int(np.argmin(sse_b6))]) # choose the split with minimum SSE.
print("best threshold:", best_thr_b6) # inspect the winning split.
assert best_thr_b6 == 2.5 # verify the intuitive split.
plt.figure(figsize=(4, 3)) # create an SSE curve.
plt.plot(thresholds_b6, sse_b6, marker="o", color="purple") # plot residual SSE by threshold.
plt.axvline(best_thr_b6, color="red", linestyle="--") # mark the selected split.
plt.title("Basic 6: choose lowest residual SSE") # title the plot.
plt.xlabel("threshold"); plt.ylabel("SSE"); plt.show() # label and display.

▶ What you'll see: the SSE curve bottoms out at threshold 2.5.

👀 Takeaway: fitting a weak learner means minimizing loss on the current pseudo-residual targets.

### Basic 7 — Add a stump with shrinkage

**Goal.** Apply $F_t=F_{t-1}+\eta h_t$, because GBMs are additive models with a learning-rate knob. We build it in 3 steps.

In [ ]:
x_b7 = np.array([0., 1., 2., 3., 4., 5.]) # define feature positions.
y_b7 = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2]) # define targets.
F0_b7 = np.full_like(y_b7, np.mean(y_b7), dtype=float) # start from the constant model.
h_b7 = np.where(x_b7 <= 2.5, -1.1833333333, 1.1833333333) # use the first stump correction.
eta_b7 = 0.4 # choose shrinkage.
print("eta:", eta_b7) # inspect the learning rate.

▶ What you'll see: the new tree will be scaled to 40% strength.

In [ ]:
F1_b7 = F0_b7 + eta_b7 * h_b7 # update predictions additively.
print("F0:", np.round(F0_b7, 3)) # inspect before.
print("F1:", np.round(F1_b7, 3)) # inspect after.
assert round(float(F1_b7[0]), 3) == 2.077 and round(float(F1_b7[-1]), 3) == 3.023 # verify the two plateau values.

In [ ]:
plt.figure(figsize=(4, 3)) # create a before-after plot.
plt.scatter(x_b7, y_b7, color="black", label="target") # show targets.
plt.plot(x_b7, F0_b7, color="gray", label="F0") # show old ensemble.
plt.step(x_b7, F1_b7, where="mid", color="seagreen", label="F1") # show updated ensemble.
plt.title("Basic 7: additive shrinkage update") # title the plot.
plt.legend(); plt.show() # display.

▶ What you'll see: predictions move downward on the left and upward on the right.

👀 Takeaway: shrinkage controls how much each weak learner can change the ensemble.

### Basic 8 — Measure empirical risk before and after

**Goal.** Average per-example losses, because the training objective is an empirical risk. We build it in 2 steps.

In [ ]:
y_b8 = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2]) # define targets.
F0_b8 = np.full_like(y_b8, np.mean(y_b8), dtype=float) # constant predictions.
F1_b8 = F0_b8 + 0.4 * np.array([-1.1833333333, -1.1833333333, -1.1833333333, 1.1833333333, 1.1833333333, 1.1833333333]) # one shrunken stump.
losses0_b8 = (y_b8 - F0_b8) ** 2 # per-example squared losses before update.
losses1_b8 = (y_b8 - F1_b8) ** 2 # per-example squared losses after update.
print("losses before:", np.round(losses0_b8, 3)) # inspect raw loss terms.
print("losses after:", np.round(losses1_b8, 3)) # inspect updated loss terms.

▶ What you'll see: most losses shrink after the stump correction.

In [ ]:
risk0_b8 = float(np.mean(losses0_b8)) # empirical risk before update.
risk1_b8 = float(np.mean(losses1_b8)) # empirical risk after update.
print("risk before:", round(risk0_b8, 3), "risk after:", round(risk1_b8, 3)) # compare averages.
assert risk1_b8 < risk0_b8 # verify improvement.
plt.figure(figsize=(4, 3)) # create a risk comparison plot.
plt.bar(["before", "after"], [risk0_b8, risk1_b8], color=["gray", "seagreen"]) # show average losses.
plt.title("Basic 8: empirical risk drops") # title the plot.
plt.ylabel("mean squared error"); plt.show() # label and display.

▶ What you'll see: the after bar is lower than the before bar.

👀 Takeaway: every boosting step should be checked by the average loss it actually reduces.

### Basic 9 — Add a cost term to the score

**Goal.** Combine raw empirical risk with a model cost, because selection should not optimize training fit alone. We build it in 2 steps.

In [ ]:
losses_b9 = np.array([0.268, 0.083, 0.539]) # verified toy losses from the lesson content.
risk_b9 = float(np.mean(losses_b9)) # empirical risk over the three examples.
cost_b9 = 0.070 # complexity, regularization, or operational cost.
print("risk:", round(risk_b9, 3), "cost:", round(cost_b9, 3)) # inspect components.
assert round(risk_b9, 3) == 0.297 # verify the raw average.

▶ What you'll see: the training-loss average is 0.297.

In [ ]:
score_b9 = risk_b9 + cost_b9 # compute the score used for model selection.
print("selection score:", round(score_b9, 3)) # inspect the full score.
assert round(score_b9, 3) == 0.367 # verify the lesson arithmetic.
plt.figure(figsize=(4, 3)) # create a component plot.
plt.bar(["risk", "cost", "score"], [risk_b9, cost_b9, score_b9], color=["steelblue", "orange", "seagreen"]) # compare pieces.
plt.title("Basic 9: score = risk + cost") # title the plot.
plt.ylabel("value"); plt.show() # display.

▶ What you'll see: the selection score is higher than the raw risk because it includes cost.

👀 Takeaway: the method's cost is part of the model-selection arithmetic, not an afterthought.

### Basic 10 — Pick the lowest decision score

**Goal.** Compare baseline, flexible, and stabilized scores on the same scale, because lower full score wins. We build it in 2 steps.

In [ ]:
baseline_b10 = 0.367 # baseline full score from risk plus cost.
flexible_b10 = 0.407 # tempting alternative score.
stable_b10 = 0.80 * baseline_b10 # stabilized score after a 20% reduction.
scores_b10 = np.array([baseline_b10, flexible_b10, stable_b10]) # collect all decision scores.
labels_b10 = np.array(["baseline", "flexible", "stabilized"]) # label the choices.
print("scores:", dict(zip(labels_b10, np.round(scores_b10, 3)))) # inspect comparable scores.

▶ What you'll see: all three candidates are expressed as decision scores where lower is better.

In [ ]:
winner_b10 = labels_b10[int(np.argmin(scores_b10))] # choose the minimum score.
print("winner:", winner_b10) # inspect selected model.
assert winner_b10 == "stabilized" # verify min(0.367, 0.407, 0.294).
plt.figure(figsize=(4, 3)) # create a decision plot.
plt.bar(labels_b10, scores_b10, color=["gray", "crimson", "seagreen"]) # show score comparison.
plt.title("Basic 10: lowest full score wins") # title the plot.
plt.ylabel("decision score"); plt.show() # display.

▶ What you'll see: the stabilized bar is lowest.

👀 Takeaway: GBM selection should compare complete scores, not isolated training fragments.

## 🟡 Easy

### Easy 1 — Run two boosting rounds from scratch

**Goal.** Repeat residual fitting and additive updates for two rounds, because a GBM is a sequence of weak corrections. We build it in 4 steps.

In [ ]:
x_e1 = np.array([0., 1., 2., 3., 4., 5.]) # define the toy feature.
y_e1 = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2]) # define targets.
F_e1 = np.full_like(y_e1, np.mean(y_e1), dtype=float) # initialize with the squared-error mean.
eta_e1 = 0.4 # choose shrinkage.
print("initial risk:", round(float(np.mean((y_e1 - F_e1) ** 2)), 3)) # inspect starting empirical risk.

▶ What you'll see: the constant model's mean squared error before any trees.

In [ ]:
risks_e1 = [float(np.mean((y_e1 - F_e1) ** 2))] # store loss curve.
trees_e1 = [] # store (threshold, left value, right value) for each stump.
for round_e1 in range(2): # fit two boosting rounds.
    residual_e1 = y_e1 - F_e1 # pseudo-residuals for squared error.
    best_e1 = None # track the best stump found this round.
    for thr_e1 in (x_e1[:-1] + x_e1[1:]) / 2: # search all split midpoints.
        left_e1 = x_e1 <= thr_e1 # route rows.
        lv_e1 = float(np.mean(residual_e1[left_e1])) # left leaf residual mean.
        rv_e1 = float(np.mean(residual_e1[~left_e1])) # right leaf residual mean.
        h_e1 = np.where(left_e1, lv_e1, rv_e1) # stump residual prediction.
        sse_e1 = float(np.sum((residual_e1 - h_e1) ** 2)) # residual SSE.
        best_e1 = (sse_e1, thr_e1, lv_e1, rv_e1, h_e1) if best_e1 is None or sse_e1 < best_e1[0] else best_e1 # keep best.
    _, thr_e1, lv_e1, rv_e1, h_e1 = best_e1 # unpack best stump.
    F_e1 = F_e1 + eta_e1 * h_e1 # add shrunken correction.
    trees_e1.append((thr_e1, lv_e1, rv_e1)) # record tree parameters.
    risks_e1.append(float(np.mean((y_e1 - F_e1) ** 2))) # record new risk.
print("trees:", [(round(t, 2), round(l, 3), round(r, 3)) for t, l, r in trees_e1]) # inspect learned stumps.

In [ ]:
print("risks:", np.round(risks_e1, 3)) # inspect monotone training improvement.
assert risks_e1[-1] < risks_e1[0] # verify boosting reduced empirical risk.

In [ ]:
plt.figure(figsize=(4, 3)) # create a learning curve.
plt.plot(range(len(risks_e1)), risks_e1, marker="o", color="teal") # plot risk after each ensemble size.
plt.title("Easy 1: risk over boosting rounds") # title the plot.
plt.xlabel("number of stumps"); plt.ylabel("MSE"); plt.show() # label and display.

▶ What you'll see: training risk falls after each additive stump.

👀 Takeaway: GBMs improve by repeatedly fitting what the current ensemble still gets wrong.

### Easy 2 — Compare learning rates for the same stump path

**Goal.** See shrinkage as a stability knob, because smaller learning rates make slower but smoother progress. We build it in 3 steps.

In [ ]:
x_e2 = np.array([0., 1., 2., 3., 4., 5.]) # define feature positions.
y_e2 = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2]) # define targets.
etas_e2 = np.array([0.2, 0.4, 1.0]) # compare cautious, moderate, and full steps.
print("learning rates:", etas_e2) # inspect candidate shrinkage values.

▶ What you'll see: three values controlling how much each tree is trusted.

In [ ]:
final_risks_e2 = [] # store final training MSE for each eta.
for eta_e2 in etas_e2: # train one tiny GBM per learning rate.
    F_e2 = np.full_like(y_e2, np.mean(y_e2), dtype=float) # reset constant start.
    for step_e2 in range(4): # use the same simple stump-fitting loop for four rounds.
        res_e2 = y_e2 - F_e2 # compute residual targets.
        best_sse_e2 = np.inf # initialize best score.
        for thr_e2 in (x_e2[:-1] + x_e2[1:]) / 2: # search all stumps.
            left_e2 = x_e2 <= thr_e2 # split rows.
            h_e2 = np.where(left_e2, np.mean(res_e2[left_e2]), np.mean(res_e2[~left_e2])) # leaf means.
            sse_e2 = float(np.sum((res_e2 - h_e2) ** 2)) # residual SSE.
            if sse_e2 < best_sse_e2: # update best stump.
                best_sse_e2, best_h_e2 = sse_e2, h_e2 # save best residual prediction.
        F_e2 = F_e2 + eta_e2 * best_h_e2 # apply shrunken update.
    final_risks_e2.append(float(np.mean((y_e2 - F_e2) ** 2))) # store final risk.
print("final risks:", np.round(final_risks_e2, 3)) # inspect fit speed.

In [ ]:
plt.figure(figsize=(4, 3)) # create a shrinkage comparison chart.
plt.bar([str(e) for e in etas_e2], final_risks_e2, color=["steelblue", "seagreen", "crimson"]) # plot final training losses.
plt.title("Easy 2: learning-rate comparison") # title the plot.
plt.xlabel("eta"); plt.ylabel("MSE after 4 stumps"); plt.show() # label and display.

▶ What you'll see: larger eta usually reduces this tiny training loss faster, but that does not automatically mean better future behavior.

👀 Takeaway: learning rate trades immediate fit speed against stability and generalization risk.

### Easy 3 — Use validation to stop adding trees

**Goal.** Track train and validation loss as trees are added, because boosting can keep fitting training residuals after validation stops improving. We build it in 4 steps.

In [ ]:
x_train_e3 = np.array([0., 1., 2., 3., 4., 5.]) # training feature positions.
y_train_e3 = np.array([1.0, 1.4, 1.7, 3.2, 3.8, 4.2]) # training targets.
x_val_e3 = np.array([0.5, 2.5, 4.5]) # validation feature positions.
y_val_e3 = np.array([1.3, 2.4, 4.1]) # validation targets.
print("train rows:", len(x_train_e3), "validation rows:", len(x_val_e3)) # inspect split sizes.

▶ What you'll see: a small held-out set will judge whether added stumps help future-like points.

In [ ]:
F_train_e3 = np.full_like(y_train_e3, np.mean(y_train_e3), dtype=float) # initialize train predictions.
F_val_e3 = np.full_like(y_val_e3, np.mean(y_train_e3), dtype=float) # initialize validation predictions with train mean.
train_curve_e3 = [float(np.mean((y_train_e3 - F_train_e3) ** 2))] # store train loss.
val_curve_e3 = [float(np.mean((y_val_e3 - F_val_e3) ** 2))] # store validation loss.
trees_e3 = [] # store stump parameters for applying to validation points.
print("initial validation MSE:", round(val_curve_e3[0], 3)) # inspect baseline validation score.

In [ ]:
for step_e3 in range(8): # add up to eight stumps.
    res_e3 = y_train_e3 - F_train_e3 # fit train residuals only.
    best_e3 = None # track best stump.
    for thr_e3 in (x_train_e3[:-1] + x_train_e3[1:]) / 2: # candidate thresholds.
        left_e3 = x_train_e3 <= thr_e3 # training split.
        lv_e3 = float(np.mean(res_e3[left_e3])); rv_e3 = float(np.mean(res_e3[~left_e3])) # leaf residual means.
        h_train_e3 = np.where(left_e3, lv_e3, rv_e3) # train correction.
        sse_e3 = float(np.sum((res_e3 - h_train_e3) ** 2)) # residual fit score.
        best_e3 = (sse_e3, thr_e3, lv_e3, rv_e3) if best_e3 is None or sse_e3 < best_e3[0] else best_e3 # keep best.
    _, thr_e3, lv_e3, rv_e3 = best_e3 # unpack chosen stump.
    F_train_e3 = F_train_e3 + 0.5 * np.where(x_train_e3 <= thr_e3, lv_e3, rv_e3) # update train predictions.
    F_val_e3 = F_val_e3 + 0.5 * np.where(x_val_e3 <= thr_e3, lv_e3, rv_e3) # apply same tree to validation.
    train_curve_e3.append(float(np.mean((y_train_e3 - F_train_e3) ** 2))) # record train MSE.
    val_curve_e3.append(float(np.mean((y_val_e3 - F_val_e3) ** 2))) # record validation MSE.
print("validation curve:", np.round(val_curve_e3, 3)) # inspect held-out behavior.

In [ ]:
best_round_e3 = int(np.argmin(val_curve_e3)) # choose ensemble size by validation loss.
print("best validation round:", best_round_e3) # inspect early-stopping point.
assert best_round_e3 >= 0 # self-check that selection is defined.

In [ ]:
plt.figure(figsize=(5, 3)) # create an early-stopping curve.
plt.plot(train_curve_e3, marker="o", label="train") # plot training loss.
plt.plot(val_curve_e3, marker="s", label="validation") # plot validation loss.
plt.axvline(best_round_e3, color="red", linestyle="--", label="best val") # mark selected round.
plt.title("Easy 3: validation chooses ensemble size") # title the plot.
plt.xlabel("number of stumps"); plt.ylabel("MSE"); plt.legend(); plt.show() # label and display.

▶ What you'll see: the train curve tends downward; the validation curve identifies the safer stopping point.

👀 Takeaway: validation loss decides whether another tree's training improvement is actually useful.

### Easy 4 — Compute the lesson's gap numbers

**Goal.** Compare a baseline score with a flexible alternative, because the score gap is the evidence for choosing one setting. We build it in 3 steps.

In [ ]:
baseline_e4 = 0.367 # full score from empirical risk plus cost.
alternative_e4 = 0.407 # flexible alternative's full score.
gap_e4 = alternative_e4 - baseline_e4 # absolute score difference.
print("baseline:", baseline_e4, "alternative:", alternative_e4) # inspect comparable scores.

▶ What you'll see: both candidates are already on the same decision-score scale.

In [ ]:
relative_gap_e4 = gap_e4 / alternative_e4 # scale the difference by the alternative score.
print("gap:", round(gap_e4, 3), "relative gap:", round(relative_gap_e4, 3)) # inspect evidence strength.
assert round(gap_e4, 3) == 0.040 and round(relative_gap_e4, 3) == 0.098 # verify lesson numbers.

In [ ]:
plt.figure(figsize=(4, 3)) # create a gap plot.
plt.bar(["baseline", "alternative"], [baseline_e4, alternative_e4], color=["seagreen", "crimson"]) # compare scores.
plt.title("Easy 4: score gap") # title the plot.
plt.ylabel("decision score"); plt.show() # display.

▶ What you'll see: the alternative is worse by 0.040, about 9.8% of its own score.

👀 Takeaway: a lower score wins, but the size of the gap tells how robust that preference may be.

### Easy 5 — Stabilize a score with shrinkage

**Goal.** Apply the lesson's 20% stabilizing reduction, because constrained models can produce better decision scores than more flexible ones. We build it in 3 steps.

In [ ]:
score_e5 = 0.367 # baseline score before stabilization.
reduction_e5 = 0.20 # stabilizing knob reduces the score by 20%.
multiplier_e5 = 1 - reduction_e5 # keep 80% of the original score.
print("multiplier:", multiplier_e5) # inspect the stabilizing factor.

▶ What you'll see: a 20% reduction means multiplying by 0.8.

In [ ]:
stable_e5 = multiplier_e5 * score_e5 # compute stabilized score.
print("stabilized score:", round(stable_e5, 3)) # inspect the lesson result.
assert round(stable_e5, 3) == 0.294 # verify 0.80 * 0.367.

In [ ]:
candidates_e5 = np.array([score_e5, 0.407, stable_e5]) # baseline, flexible alternative, stabilized score.
labels_e5 = ["baseline", "flexible", "stabilized"] # label choices.
plt.figure(figsize=(4, 3)) # create a final comparison plot.
plt.bar(labels_e5, candidates_e5, color=["gray", "crimson", "seagreen"]) # compare full scores.
plt.title("Easy 5: stabilization can win") # title the plot.
plt.ylabel("decision score"); plt.show() # display.

▶ What you'll see: the stabilized score is below both the baseline and the flexible alternative.

👀 Takeaway: the best GBM setting is the one with the lowest full decision score, not necessarily the most flexible fit.

## 🔴 Advanced

### Advanced 1 — Implement reusable stump boosting

**Goal.** Package the stump search into a small loop, because a GBM is a repeated residual-fitting algorithm. We build it in 4 steps.

In [ ]:
x_a1 = np.linspace(0, 1, 12) # create a small feature grid.
y_a1 = np.sin(2 * np.pi * x_a1) + 0.3 * x_a1 # define a smooth target with shape a stump ensemble can approximate.
F_a1 = np.full_like(y_a1, np.mean(y_a1), dtype=float) # initialize predictions with the mean.
print("initial MSE:", round(float(np.mean((y_a1 - F_a1) ** 2)), 3)) # inspect starting loss.

▶ What you'll see: a nonzero starting error from a flat prediction.

In [ ]:
def best_stump_a1(x, residual): # find the best one-split residual learner for squared error.
    best = None # store best (SSE, threshold, left value, right value, prediction).
    for thr in (x[:-1] + x[1:]) / 2: # candidate split points.
        left = x <= thr # route examples left.
        lv = float(np.mean(residual[left])); rv = float(np.mean(residual[~left])) # leaf means.
        pred = np.where(left, lv, rv) # residual prediction.
        sse = float(np.sum((residual - pred) ** 2)) # fit error.
        best = (sse, thr, lv, rv, pred) if best is None or sse < best[0] else best # keep minimum SSE.
    return best # return the best stump details.
print("helper ready") # confirm the function is defined.

In [ ]:
losses_a1 = [float(np.mean((y_a1 - F_a1) ** 2))] # store loss curve.
for t_a1 in range(10): # add ten weak learners.
    residual_a1 = y_a1 - F_a1 # current negative-gradient target.
    _, thr_a1, lv_a1, rv_a1, h_a1 = best_stump_a1(x_a1, residual_a1) # fit best stump.
    F_a1 = F_a1 + 0.3 * h_a1 # add shrunken correction.
    losses_a1.append(float(np.mean((y_a1 - F_a1) ** 2))) # record empirical risk.
print("first/final MSE:", round(losses_a1[0], 3), round(losses_a1[-1], 3)) # inspect improvement.
assert losses_a1[-1] < losses_a1[0] # verify training loss fell.

In [ ]:
plt.figure(figsize=(5, 3)) # create a fitted-function plot.
plt.scatter(x_a1, y_a1, color="black", label="target") # show data.
plt.step(x_a1, F_a1, where="mid", color="seagreen", label="10-stump GBM") # show ensemble prediction.
plt.title("Advanced 1: from-scratch stump GBM") # title the plot.
plt.legend(); plt.show() # display.

▶ What you'll see: the stepwise ensemble approximates the curved target more closely than a constant.

👀 Takeaway: GBM complexity comes from many simple additive corrections, not from one complicated learner.

### Advanced 2 — Track validation while training

**Goal.** Use a validation curve to choose the number of boosting rounds, because training loss alone rewards ever more flexibility. We build it in 4 steps.

In [ ]:
x_train_a2 = np.linspace(0, 1, 12) # training feature grid.
y_train_a2 = np.sin(2 * np.pi * x_train_a2) + 0.3 * x_train_a2 # training targets.
x_val_a2 = np.linspace(0.05, 0.95, 10) # validation feature grid shifted between training points.
y_val_a2 = np.sin(2 * np.pi * x_val_a2) + 0.3 * x_val_a2 # validation targets from the same function.
print("train/val sizes:", len(x_train_a2), len(x_val_a2)) # inspect split sizes.

▶ What you'll see: validation points are distinct from training points.

In [ ]:
def stump_predict_a2(x, thr, lv, rv): # apply a learned stump to any feature array.
    return np.where(x <= thr, lv, rv) # left rows get lv, right rows get rv.
F_train_a2 = np.full_like(y_train_a2, np.mean(y_train_a2), dtype=float) # train baseline.
F_val_a2 = np.full_like(y_val_a2, np.mean(y_train_a2), dtype=float) # validation baseline uses train mean.
train_loss_a2 = [float(np.mean((y_train_a2 - F_train_a2) ** 2))] # initial train loss.
val_loss_a2 = [float(np.mean((y_val_a2 - F_val_a2) ** 2))] # initial validation loss.
print("initial losses:", round(train_loss_a2[0], 3), round(val_loss_a2[0], 3)) # inspect baseline losses.

In [ ]:
for t_a2 in range(20): # train up to twenty rounds.
    residual_a2 = y_train_a2 - F_train_a2 # compute train pseudo-residuals.
    best_a2 = None # track best stump.
    for thr_a2 in (x_train_a2[:-1] + x_train_a2[1:]) / 2: # search train thresholds.
        left_a2 = x_train_a2 <= thr_a2 # route train rows.
        lv_a2 = float(np.mean(residual_a2[left_a2])); rv_a2 = float(np.mean(residual_a2[~left_a2])) # residual means.
        h_train_a2 = stump_predict_a2(x_train_a2, thr_a2, lv_a2, rv_a2) # train correction.
        sse_a2 = float(np.sum((residual_a2 - h_train_a2) ** 2)) # residual SSE.
        best_a2 = (sse_a2, thr_a2, lv_a2, rv_a2) if best_a2 is None or sse_a2 < best_a2[0] else best_a2 # keep best.
    _, thr_a2, lv_a2, rv_a2 = best_a2 # unpack chosen stump.
    F_train_a2 = F_train_a2 + 0.25 * stump_predict_a2(x_train_a2, thr_a2, lv_a2, rv_a2) # update train predictions.
    F_val_a2 = F_val_a2 + 0.25 * stump_predict_a2(x_val_a2, thr_a2, lv_a2, rv_a2) # update validation predictions.
    train_loss_a2.append(float(np.mean((y_train_a2 - F_train_a2) ** 2))) # record train MSE.
    val_loss_a2.append(float(np.mean((y_val_a2 - F_val_a2) ** 2))) # record validation MSE.
print("best validation MSE:", round(float(np.min(val_loss_a2)), 3)) # inspect best held-out score.

In [ ]:
best_round_a2 = int(np.argmin(val_loss_a2)) # select by validation loss.
plt.figure(figsize=(5, 3)) # create validation diagnostic.
plt.plot(train_loss_a2, label="train", color="steelblue") # plot training curve.
plt.plot(val_loss_a2, label="validation", color="orange") # plot validation curve.
plt.axvline(best_round_a2, color="red", linestyle="--", label=f"best={best_round_a2}") # mark early stop.
plt.title("Advanced 2: validation curve for GBM rounds") # title the plot.
plt.xlabel("round"); plt.ylabel("MSE"); plt.legend(); plt.show() # label and display.

▶ What you'll see: validation selects a finite number of rounds instead of blindly trusting the longest ensemble.

👀 Takeaway: the validation curve is the practical check against boosting past reusable structure.

### Advanced 3 — Compare depth by cost-adjusted score

**Goal.** Penalize more flexible learners, because deeper or richer trees should justify their extra capacity with enough validation gain. We build it in 3 steps.

In [ ]:
train_loss_a3 = np.array([0.210, 0.160, 0.120]) # hypothetical training losses for depths 1, 2, and 3.
val_loss_a3 = np.array([0.268, 0.252, 0.249]) # validation losses improve only slightly with depth.
cost_a3 = np.array([0.020, 0.055, 0.095]) # complexity costs grow with depth.
depths_a3 = np.array([1, 2, 3]) # model flexibility levels.
print("validation losses:", val_loss_a3) # inspect raw validation fit.

▶ What you'll see: deeper trees look slightly better before adding cost.

In [ ]:
score_a3 = val_loss_a3 + cost_a3 # compute cost-adjusted decision score.
best_depth_a3 = int(depths_a3[np.argmin(score_a3)]) # choose the lowest full score.
print("cost-adjusted scores:", np.round(score_a3, 3)) # inspect final scores.
print("best depth:", best_depth_a3) # inspect selected complexity.
assert best_depth_a3 == 1 # depth 1 wins after cost in this toy setup.

In [ ]:
plt.figure(figsize=(5, 3)) # create a depth comparison plot.
plt.plot(depths_a3, val_loss_a3, marker="o", label="validation loss") # raw held-out loss.
plt.plot(depths_a3, score_a3, marker="s", label="loss + cost") # full selection score.
plt.title("Advanced 3: flexibility must pay its cost") # title the plot.
plt.xlabel("tree depth"); plt.ylabel("score"); plt.legend(); plt.show() # label and display.

▶ What you'll see: the raw validation curve and cost-adjusted curve can prefer different depths.

👀 Takeaway: a more flexible GBM setting should win only if its gain survives the relevant penalty.

### Advanced 4 — Simulate subsampling for stability

**Goal.** Compare full-data and subsampled residual fits, because stochastic boosting can reduce brittle dependence on any one sample. We build it in 4 steps.

In [ ]:
x_a4 = np.linspace(0, 1, 10) # define a small feature grid.
y_a4 = np.sin(2 * np.pi * x_a4) + 0.2 * np.arange(10) / 9 # define a target with trend and curvature.
F_a4 = np.full_like(y_a4, np.mean(y_a4), dtype=float) # start from mean prediction.
res_a4 = y_a4 - F_a4 # compute residual targets.
print("residual mean:", round(float(np.mean(res_a4)), 10)) # mean residual after constant start is zero.

▶ What you'll see: residuals are centered, so stumps model structure rather than a global offset.

In [ ]:
idx_full_a4 = np.arange(len(x_a4)) # use every row for full-data fitting.
idx_sub_a4 = np.array([0, 1, 3, 5, 7, 9]) # deterministic subsample for the stochastic fit.
print("subsample indices:", idx_sub_a4) # inspect which rows drive the stochastic learner.

In [ ]:
def fit_stump_on_indices_a4(x, residual, indices): # fit a stump using only selected rows.
    best = None # track best candidate.
    for thr in (x[:-1] + x[1:]) / 2: # candidate thresholds on the global feature grid.
        left_all = x <= thr # routing for all rows.
        left = left_all[indices] # routing for selected rows.
        if np.sum(left) == 0 or np.sum(~left) == 0: # skip splits that leave an empty sampled leaf.
            continue # empty leaf has no mean residual.
        lv = float(np.mean(residual[indices][left])); rv = float(np.mean(residual[indices][~left])) # sampled leaf means.
        pred_sample = np.where(left, lv, rv) # predictions on selected rows.
        sse = float(np.sum((residual[indices] - pred_sample) ** 2)) # sampled residual SSE.
        best = (sse, thr, lv, rv) if best is None or sse < best[0] else best # keep best.
    return best # return stump parameters.
full_a4 = fit_stump_on_indices_a4(x_a4, res_a4, idx_full_a4) # fit on all rows.
sub_a4 = fit_stump_on_indices_a4(x_a4, res_a4, idx_sub_a4) # fit on the subsample.
print("full stump:", (round(full_a4[1], 3), round(full_a4[2], 3), round(full_a4[3], 3))) # inspect full-data learner.
print("subsample stump:", (round(sub_a4[1], 3), round(sub_a4[2], 3), round(sub_a4[3], 3))) # inspect stochastic learner.

In [ ]:
h_full_a4 = np.where(x_a4 <= full_a4[1], full_a4[2], full_a4[3]) # apply full-data stump to all rows.
h_sub_a4 = np.where(x_a4 <= sub_a4[1], sub_a4[2], sub_a4[3]) # apply subsampled stump to all rows.
plt.figure(figsize=(5, 3)) # create a stochastic comparison plot.
plt.plot(x_a4, res_a4, marker="o", color="black", label="residual") # plot residual targets.
plt.step(x_a4, h_full_a4, where="mid", label="full-data stump") # plot full fit.
plt.step(x_a4, h_sub_a4, where="mid", label="subsample stump") # plot sampled fit.
plt.title("Advanced 4: subsampling changes one correction") # title the plot.
plt.legend(); plt.show() # display.

▶ What you'll see: the subsampled stump may choose a different threshold or leaf values, injecting controlled randomness.

👀 Takeaway: stochastic GBM variants trade a little single-step certainty for more stable ensembles across many steps.

### Advanced 5 — End-to-end score table for GBM choices

**Goal.** Build a final selection table with risk, cost, gap, and stabilized score, because the lesson's decision rule is a complete comparison. We build it in 4 steps.

In [ ]:
names_a5 = np.array(["baseline", "flexible", "stabilized"]) # define candidate GBM settings.
raw_risk_a5 = np.array([0.297, 0.327, 0.235]) # raw empirical or validation risk terms.
cost_a5 = np.array([0.070, 0.080, 0.059]) # costs or penalties for the settings.
print("candidates:", names_a5) # inspect the rows of the table.

▶ What you'll see: three candidate settings will be compared on the same scale.

In [ ]:
score_a5 = raw_risk_a5 + cost_a5 # compute full decision scores.
print("scores:", dict(zip(names_a5, np.round(score_a5, 3)))) # inspect score table.
assert round(float(score_a5[0]), 3) == 0.367 and round(float(score_a5[1]), 3) == 0.407 and round(float(score_a5[2]), 3) == 0.294 # verify lesson scores.

In [ ]:
gap_to_flexible_a5 = score_a5[1] - score_a5[0] # baseline advantage over flexible alternative.
relative_gap_a5 = gap_to_flexible_a5 / score_a5[1] # normalize gap by alternative score.
best_a5 = names_a5[int(np.argmin(score_a5))] # choose the lowest full score.
print("gap baseline vs flexible:", round(gap_to_flexible_a5, 3), "relative:", round(relative_gap_a5, 3)) # inspect gap evidence.
print("selected setting:", best_a5) # inspect final decision.
assert best_a5 == "stabilized" # stabilized score is lowest.

In [ ]:
plt.figure(figsize=(5, 3)) # create a stacked score chart.
plt.bar(names_a5, raw_risk_a5, label="risk", color="steelblue") # draw raw fit component.
plt.bar(names_a5, cost_a5, bottom=raw_risk_a5, label="cost", color="orange") # stack cost component.
plt.title("Advanced 5: complete GBM decision scores") # title the plot.
plt.ylabel("risk + cost"); plt.legend(); plt.show() # label and display.

▶ What you'll see: the stabilized model has the smallest total bar even though each component matters.

👀 Takeaway: final GBM selection is arithmetic discipline: average the relevant loss, add the cost, compare gaps, and carry forward the lowest stable score.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Gradient boosting fits each new learner to the loss gradient left by the current ensemble.

Gradient Boosting Machines turn empirical-risk minimization into a stagewise additive procedure. The notebook proves the lesson's arithmetic, then fits real gradient-boosted trees rather than a cartoon residual sketch. Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)



def _fit_predict(model, x_tr, y_tr, x_te):
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def _stump_adaboost(n_estimators=60, learning_rate=0.6):
    stump = DecisionTreeClassifier(max_depth=1, random_state=7)
    try:
        return AdaBoostClassifier(estimator=stump, n_estimators=n_estimators, learning_rate=learning_rate, random_state=7)
    except TypeError:
        return AdaBoostClassifier(base_estimator=stump, n_estimators=n_estimators, learning_rate=learning_rate, random_state=7)


def _project2d(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 1:
        return np.c_[X[:, 0], np.zeros(X.shape[0])]
    return X[:, :2]


def _plot_regions(ax, model, X, y, title):
    x2 = _project2d(X)
    scaler = StandardScaler()
    xs = scaler.fit_transform(x2)
    fitted = clone(model)
    try:
        fitted.fit(xs, y)
    except ValueError:
        fitted = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        fitted.fit(xs, y)
    x_min = xs[:, 0].min() - 0.8
    x_max = xs[:, 0].max() + 0.8
    y_min = xs[:, 1].min() - 0.8
    y_max = xs[:, 1].max() + 0.8
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 80), np.linspace(y_min, y_max, 80))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = fitted.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="tab10")
    ax.scatter(xs[:, 0], xs[:, 1], c=y, s=16, cmap="tab10", edgecolor="k", linewidth=0.2)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


def summarize_ladder(rungs):
    for name, X, y in rungs:
        classes, counts = np.unique(y, return_counts=True)
        print(f"{name}: X={X.shape}, classes={dict(zip(classes.tolist(), counts.tolist()))}")
    sample_name, sample_X, sample_y = rungs[0]
    print("sample rung:", sample_name)
    print(np.c_[sample_X, sample_y][:4])


def run_ladder(build_and_predict):
    rows = []
    for i, (name, X, y) in enumerate(clf_ladder(), start=1):
        acc = clf_accuracy(build_and_predict, X, y)
        rows.append((i, name, float(acc)))
    print("rung | accuracy | dataset")
    for i, name, acc in rows:
        print(f"D{i} | {acc:.3f} | {name}")
    return rows


def plot_summary(rows, model_factory, title):
    rungs = clf_ladder()
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    axes = axes.ravel()
    for ax, (name, X, y) in zip(axes[:5], rungs):
        _plot_regions(ax, model_factory(), X, y, name.split("(")[0])
    axes[5].plot([r[0] for r in rows], [r[2] for r in rows], marker="o")
    axes[5].set_ylim(0.0, 1.05)
    axes[5].set_xlabel("ladder rung")
    axes[5].set_ylabel("held-out accuracy")
    axes[5].set_title(title)
    axes[5].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def lesson_score(losses, cost, alternative):
    empirical = round(sum(losses) / len(losses), 3)
    score = round(empirical + cost, 3)
    gap = round(alternative - score, 3)
    return empirical, score, gap


def cost_sensitive_choice(raw_loss, cost, competitor):
    score = raw_loss + cost
    return score, score < competitor


## The concept, built once (D1)

The lesson formula is $$F_t(x)=F_{t-1}(x)+\eta h_t(x)$$. We first rebuild the tiny score: average the three lesson losses, add the cost, and assert the exact plan numbers.

In [ ]:

def gradient_boosting_machines_method():
    losses = np.array([0.268, 0.083, 0.539], dtype=float)
    empirical, score, gap = lesson_score(losses, 0.07, 0.407)
    assert empirical == 0.297
    assert score == 0.367
    assert gap == 0.04
    f0 = np.array([0.10, -0.20, 0.05])
    residual_tree = np.array([0.40, -0.10, 0.20])
    eta = 0.25
    f1 = f0 + eta * residual_tree
    return {"empirical": empirical, "score": score, "gap": gap, "updated_scores": f1}

result = gradient_boosting_machines_method()
print(result)


The printed dictionary contains the hand-checkable lesson score plus one method-specific quantity: a vote weight, an additive update, a second-order surrogate, a blend, a margin objective, or a kernel identity.

In [ ]:
checked = gradient_boosting_machines_method()
assert checked['score'] == 0.367
print('D1 arithmetic verified for 3.26')

## The dataset ladder

All classification notebooks use the shared `clf_ladder()` and `clf_accuracy()` helpers embedded above, so the notebook is self-contained in Colab.

In [ ]:
rungs = clf_ladder()
summarize_ladder(rungs)

## Run the same method across D1-D5

The metric is held-out accuracy. Macro-F1 would be a useful companion when class skew is severe, especially on D5.

In [ ]:


def build_and_predict(x_tr, y_tr, x_te):
    model = GradientBoostingClassifier(n_estimators=80, learning_rate=0.08, max_depth=2, random_state=7)
    return _fit_predict(model, x_tr, y_tr, x_te)

rows = run_ladder(build_and_predict)
assert len(rows) == 5
assert all(0.0 <= acc <= 1.0 for _, _, acc in rows)


## Results visualization

The small multiples show the learned artifact on the first two standardized features for every rung; the summary panel tracks accuracy as the data become more realistic.

In [ ]:
plot_summary(rows, lambda: GradientBoostingClassifier(n_estimators=80, learning_rate=0.08, max_depth=2, random_state=7), 'Gradient Boosting Machines accuracy')

## Pitfall on D5: optimizing the raw term and forgetting the cost

On D5, too many trees can lower the raw validation loss only marginally while carrying extra complexity. The lesson's cost-aware score rejects that overfit-looking choice. The wrong behavior ranks by raw validation loss; the fix ranks by the lesson score with cost and the validation-gap scale.

In [ ]:

X, y = clf_ladder()[-1][1:]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)
lean = GradientBoostingClassifier(n_estimators=80, learning_rate=0.08, max_depth=2, random_state=7)
large = GradientBoostingClassifier(n_estimators=80, learning_rate=0.08, max_depth=2, random_state=7)
if hasattr(large, "set_params"):
    params = large.get_params()
    if "n_estimators" in params:
        large.set_params(n_estimators=min(params["n_estimators"] * 3, 240))
    if "max_iter" in params and params["max_iter"] > 0:
        large.set_params(max_iter=min(params["max_iter"] * 3, 240))
    if "C" in params:
        large.set_params(C=25.0)
    if "gamma" in params:
        large.set_params(gamma=3.0)
lean.fit(x_tr, y_tr)
large.fit(x_tr, y_tr)
lean_loss = 1.0 - accuracy_score(y_te, lean.predict(x_te))
large_loss = 1.0 - accuracy_score(y_te, large.predict(x_te))
wrong_pick = "large" if large_loss <= lean_loss else "lean"
lean_score, lean_ok = cost_sensitive_choice(lean_loss, 0.07, large_loss + 0.07 + 0.04)
large_score = large_loss + 0.07 + 0.04
fixed_pick = "lean" if lean_score <= large_score else "large"
print("raw losses:", {"lean": round(lean_loss, 3), "large": round(large_loss, 3)})
print("wrong raw-loss pick:", wrong_pick)
print("cost-aware scores:", {"lean": round(lean_score, 3), "large": round(large_score, 3)})
print("fixed pick:", fixed_pick)
assert lean_score <= large_score or fixed_pick == "large"


## Evaluate it + Practice

- Compare held-out accuracy against a majority-class no-skill baseline.
- Sanity check that shuffling labels pushes accuracy toward chance.
- Ablate the key idea: fewer boosting rounds, no honest stacking, linear instead of kernel, or tiny/huge C should change the metric.
- Watch failure signals: unstable D5 score, perfect train accuracy with weak validation accuracy, or a cost-aware score that reverses the raw-loss winner.

Practice 1: change one hyperparameter and re-run the D1-D5 table.

Practice 2: add a majority-class baseline row for every rung.

Practice 3: repeat D5 with a different random split and compare the validation gap.